In [197]:
import json
import os
import pandas as pd
import re
import shutil
from collections import defaultdict
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [198]:
#model = "Llama-3-8B-Instruct_3_shot"
model = "Llama-3-70B-Instruct_3_shot"
p2_label_path = "chia_label/p2"
ready_path = f"model_output/{model}/ready"
failed_model_path = f"model_output/{model}/failed_inner"
p2_model_formatted_path = f"model_output/{model}/formatted"
eval_path = f"evaluate/batch1"

for path in [ready_path, failed_model_path, p2_model_formatted_path]: 
    os.makedirs(path, exist_ok=True)

In [199]:
def extract_nct_number(filename):
    parts = filename.split('_')
    nct_number = None
    file_type = None
    for part in parts:
        if part.startswith("NCT"):
            nct_number = part
        if part in ["inc", "exc"]:
            file_type = part
    return nct_number+"_"+file_type

def read_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return json.load(file)


def extract_logical_structure(data):
    structure = defaultdict(int)
    def traverse(node, depth=0):
        nonlocal structure
        structure["depth"] = max(structure["depth"], depth)

        if isinstance(node, dict):
            for key in node:
                if key in ["AND", "OR", "NOT"]:
                    structure[key] += 1
                traverse(node[key], depth + 1)
        elif isinstance(node, list):
            for item in node:
                traverse(item, depth + 1)

    traverse(data)
    return dict(structure)

In [200]:
label_files = {extract_nct_number(f): os.path.join(p2_label_path, f) for f in os.listdir(p2_label_path) if f.endswith('.json')}
model_files = {extract_nct_number(f): os.path.join(p2_model_formatted_path, f) for f in os.listdir(p2_model_formatted_path) if f.endswith('.json')}

common_ncts = set(label_files.keys()).intersection(model_files.keys())

In [215]:
labels = []
predictions = []
success_data = []

In [216]:
for nct in common_ncts:# ["NCT00050349_exc"]: #
    try:
        label_data = read_json(label_files[nct])
        model_data = read_json(model_files[nct])

        label_structure = extract_logical_structure(label_data)
        model_structure = extract_logical_structure(model_data)
        success_data.append({
            'NCT': nct,
            'label_AND': label_structure.get('AND', 0),
            'label_OR': label_structure.get('OR', 0),
            'label_NOT': label_structure.get('NOT', 0),
            'label_DEPTH': label_structure.get('depth', 0),
            'model_AND': model_structure.get('AND', 0),
            'model_OR': model_structure.get('OR', 0),
            'model_NOT': model_structure.get('NOT', 0),
            'model_DEPTH': model_structure.get('depth', 0),
            'diff_AND': 1 if model_structure.get('AND', 0) > label_structure.get('AND', 0) else -1 if model_structure.get('AND', 0) < label_structure.get('AND', 0) else 0,
            'diff_OR': 1 if model_structure.get('OR', 0) > label_structure.get('OR', 0) else -1 if model_structure.get('OR', 0) < label_structure.get('OR', 0) else 0,
            'diff_NOT': 1 if model_structure.get('NOT', 0) > label_structure.get('NOT', 0) else -1 if model_structure.get('NOT', 0) < label_structure.get('NOT', 0) else 0,
            'diff_DEPTH': 1 if model_structure.get('depth', 0) > label_structure.get('depth', 0) else -1 if model_structure.get('depth', 0) < label_structure.get('depth', 0) else 0
            
        })

        labels.append(label_structure)
        predictions.append(model_structure)
        #shutil.copy(model_files[nct], os.path.join(ready_path, os.path.basename(model_files[nct])))
    except Exception as e:
        print(f"Error processing NCT {nct}: {e}")
        #shutil.copy(model_files[nct], os.path.join(failed_model_path, os.path.basename(model_files[nct])))

Error processing NCT NCT01701219_inc: Extra data: line 63 column 1 (char 1645)
Error processing NCT NCT00650312_inc: Expecting ',' delimiter: line 91 column 173 (char 3815)


In [218]:
df_success = pd.DataFrame(success_data).set_index('NCT')
#df_success.to_csv(eval_path+f'/{model}_eval.csv')

df_success

,label_AND,label_OR,label_NOT,label_DEPTH,model_AND,model_OR,model_NOT,model_DEPTH,diff_AND,diff_OR,diff_NOT,diff_DEPTH
NCT,,,,,,,,,,,,
NCT00752310_exc,7,12,2,19,5,2,6,15,-1,-1,1,-1
NCT01349413_exc,14,3,0,23,8,1,0,17,-1,-1,0,-1
NCT00397215_inc,4,2,0,11,4,1,0,9,0,-1,0,-1
NCT01650792_inc,3,0,0,7,2,0,0,5,-1,0,0,-1
NCT01567605_exc,11,5,2,23,9,3,3,19,-1,-1,1,-1
...,...,...,...,...,...,...,...,...,...,...,...,...
NCT01614041_exc,14,9,0,29,14,6,0,29,0,-1,0,0
NCT00609531_inc,8,3,2,27,6,3,2,21,-1,0,0,-1
NCT00312429_inc,8,4,0,17,8,0,0,17,0,-1,0,0


In [221]:
true_values = df_success[['label_AND', 'label_OR', 'label_NOT', 'label_DEPTH']].values
predicted_values = df_success[['model_AND', 'model_OR', 'model_NOT', 'model_DEPTH']].values

metrics = {}
for i, metric in enumerate(['AND', 'OR', 'NOT', 'DEPTH']):
    y_true = true_values[:, i]
    y_pred = predicted_values[:, i]

    diffs = y_pred - y_true
    pct_greater = (diffs > 0).sum() / len(diffs) * 100
    pct_less = (diffs < 0).sum() / len(diffs) * 100
    pct_equal = (diffs == 0).sum() / len(diffs) * 100


    metrics[metric] = {
        'accuracy': round(accuracy_score(y_true, y_pred), 3),
        'precision': round(precision_score(y_true, y_pred, average='weighted', zero_division=0), 3),
        'recall': round(recall_score(y_true, y_pred, average='weighted', zero_division=0), 3),
        'f1_score': round(f1_score(y_true, y_pred, average='weighted', zero_division=0), 3),
        'pct_greater': round(pct_greater, 2),
        'pct_less': round(pct_less, 2),
        'pct_equal': round(pct_equal, 2)
       # 'confusion_matrix': confusion_matrix(y_true, y_pred)
    }
    metrics_df = pd.DataFrame(metrics).T  
    num_nct_files = len(df_success)
    metrics_df['num_nct_files'] = num_nct_files
    metrics_df['model_name'] = model
    # Save Metrics to CSV
    metrics_df.to_csv(os.path.join(eval_path, f'{model}_metrics_summary.csv'))

print(f"{len(df_success)} Daten mit {model}")
for metric, values in metrics.items():
    print(f"Metrics for {metric}:")
    print(f"  Accuracy: {values['accuracy']}")
    print(f"  Precision: {values['precision']}")
    print(f"  Recall: {values['recall']}")
    print(f"  F1 Score: {values['f1_score']}")

    print(f"  % Greater: {values['pct_greater']}")
    print(f"  % Less: {values['pct_less']}")
    print(f"  % Equal: {values['pct_equal']}")
    print()
    #print(f"  Confusion Matrix:\n{values['confusion_matrix']}\n")

298 Daten mit Llama-3-70B-Instruct_3_shot
Metrics for AND:
  Accuracy: 0.121
  Precision: 0.119
  Recall: 0.121
  F1 Score: 0.117
  % Greater: 16.11
  % Less: 71.81
  % Equal: 12.08

Metrics for OR:
  Accuracy: 0.379
  Precision: 0.325
  Recall: 0.379
  F1 Score: 0.332
  % Greater: 10.4
  % Less: 51.68
  % Equal: 37.92

Metrics for NOT:
  Accuracy: 0.678
  Precision: 0.632
  Recall: 0.678
  F1 Score: 0.64
  % Greater: 8.72
  % Less: 23.49
  % Equal: 67.79

Metrics for DEPTH:
  Accuracy: 0.151
  Precision: 0.166
  Recall: 0.151
  F1 Score: 0.155
  % Greater: 14.43
  % Less: 70.47
  % Equal: 15.1



In [205]:
matching_rows = df_success[df_success['label_AND'] == df_success['model_AND']][['label_AND', 'model_AND']]
matching_rows

,label_AND,model_AND
NCT,,
NCT00397215_inc,4,4
NCT01483118_exc,11,11
NCT00886158_inc,4,4
NCT01531257_inc,4,4
NCT01715714_inc,5,5
NCT01051414_exc,3,3
NCT01261832_inc,1,1
NCT01000155_inc,11,11
NCT00440245_exc,1,1
